# LEPTOLIC — ทำนายผลในอนาคตจาก NHEFS

**เปิดใน Google Colab → Runtime → Run all (CPU)**

ดาวน์โหลดข้อมูลจริง → รวมคนเดิม → สร้าง Target 8 ปี → ฝึก Logistic Regression
→ ปรับ probability ด้วยชุด Calibration → ประเมิน Test → ทดลองกรอกข้อมูล → ดาวน์โหลดผล
ใช้ NumPy ที่มีใน Colab ไม่ใช้ API เสียเงิน

### สิ่งที่ทำนาย
ผลรายงานการได้รับการวินิจฉัยความดันสูงครั้งแรก **ภายใน 8 ปี** หลังการตรวจปี 1982–1984
ใช้ผลติดตามปี 1986, 1987 และ 1992 เป้าหมายนี้ต่างจากการจำแนกความดันสูงในปัจจุบัน
ปีวินิจฉัยมาจากคำตอบสัมภาษณ์ ไม่ใช่วันที่เริ่มเกิดโรคหรือค่าความดันที่วัดในอนาคต

**ไฟล์นี้ยังไม่มีโมเดลที่ฝึกจาก NHEFS หรือคะแนนจริงแนบมา** เพราะเซสชันที่สร้างไฟล์
ดาวน์โหลด CDC ไม่ได้ โค้ดและการฝึก/ประเมินถูกทดสอบด้วยข้อมูลจำลองแล้ว
การกด Run all จะฝึกจาก NHEFS จริง และไม่ใช้ข้อมูลจำลองแทนเมื่อดาวน์โหลดล้มเหลว

### ขอบเขตของต้นแบบ
- คัดผู้ตอบว่าไม่เคยได้รับการวินิจฉัย (D2=2) และค่าความดันตั้งต้นต่ำกว่า 130/80 ตามกฎโครงงาน
- Inputs: อายุ เพศ ความดันตั้งต้น และชั่วโมงนอนเฉลี่ย ไม่ใช่ฟีเจอร์เดิม 11 ตัว
- ข้อมูล NHANES 2021–2023 ในไฟล์เดิมไม่ถูกนำมาจับคู่กับคนใน NHEFS
- ค่าว่าง/ข้ามคำถาม/ไม่รู้/ติดตามไม่ครบไม่ถูกแทนว่าไม่เกิดโรค
- ถ้าปีวินิจฉัยคร่อมวันครบ 8 ปี จะตัดกรณีนั้นออก ไม่สร้างวันที่สมมติ
- กลุ่มที่มี label ชัดอาจมี selection bias โมเดลนี้ไม่ได้แก้ informative censoring
- ผลเป็น **คะแนนทดลองจากกลุ่มตัวอย่างสหรัฐฯ ในอดีต** ยังไม่ยืนยันสำหรับคนไทยยุคนี้

คู่มือและแหล่งข้อมูล:
https://wwwn.cdc.gov/nchs/nhanes/nhefs/default.aspx


In [ ]:
"""NHEFS historical 8-year first-reported-diagnosis prototype; NumPy only."""
from pathlib import Path
from urllib.request import Request, urlopen
from datetime import date
from collections import Counter
import csv, hashlib, json, math, re, zipfile
import numpy as np

BASE='https://wwwn.cdc.gov/nchs/data/nhefs/'
HORIZON_YEARS=8
SEED=42
FEATURES=['age_years','sex_female','baseline_sbp','baseline_dbp','sleep_hours']
WAVES={
 '82':{'data':'N82int.txt','layout':'intv82.inputs.labels.txt','id':'SEQNUM1',
 'fields':['SEQNUM1','SUBAGE','SUBSEX','D2','D3','FINTDMO','FINTDDY','FINTDYR',
 'PMR4cAVS','PMR4cAVD','PMR4c2SY','PMR4c2DI','PMR4c3SY','PMR4c3DI','OH7',
 'N56','N58AMT','N58PER']},
 '86':{'data':'N86int.txt','layout':'intv86.inputs.labels.txt','id':'SEQNO',
 'fields':['SEQNO','HYPER','HBPYEAR1','BASERSLT','FINTMO','FINTDAY','FINTYR','DEATHYR']},
 '87':{'data':'N87int.txt','layout':'intv87.inputs.labels.txt','id':'SEQNO',
 'fields':['SEQNO','HYPER','HBPYEAR1','BLOODPRE','BASERSLT','RSLTMO','RSLTDAY','RSLTYR','DEATHYR']},
 '92':{'data':'N92int.txt','layout':'intv92.inputs.labels.txt','id':'HANESEQ',
 'fields':['HANESEQ','HYPER','HBPYEAR1','LHIGHBP','B49DISF','MAINRSLT',
 'RSLTMO','RSLTDY','RSLTYR','NEWDODYR']}}

def integer(value):
    try:return int(str(value).strip())
    except (ValueError,TypeError):return None

def valid_number(value,lo,hi):
    try:v=float(str(value).strip())
    except (ValueError,TypeError):return np.nan
    return v if np.isfinite(v) and lo<=v<=hi else np.nan

def interview_date(row,year,month,day,lo,hi):
    y,m,d=integer(row.get(year)),integer(row.get(month)),integer(row.get(day))
    if y is not None and 0<=y<=99:y+=1900
    if y is None or not lo<=y<=hi:return None
    try:return date(y,m,d)
    except (ValueError,TypeError):return None

def anniversary(start,years):
    try:return start.replace(year=start.year+years)
    except ValueError:return start.replace(year=start.year+years,day=28)

def bp_component(row,summary,second,third,lo,hi):
    avg=valid_number(row.get(summary),lo,hi)
    if np.isfinite(avg):return avg
    values=[valid_number(row.get(k),lo,hi) for k in (second,third)]
    return float(np.mean(values)) if all(np.isfinite(values)) else np.nan

def baseline_features(row):
    return {
     'age_years':valid_number(row.get('SUBAGE'),18,100),
     'sex_female':{1:0.,2:1.}.get(integer(row.get('SUBSEX')),np.nan),
     'baseline_sbp':bp_component(row,'PMR4cAVS','PMR4c2SY','PMR4c3SY',60,300),
     'baseline_dbp':bp_component(row,'PMR4cAVD','PMR4c2DI','PMR4c3DI',30,180),
     # Numeric physical validity filter; special 97/98/99 codes are not hours.
     'sleep_hours':valid_number(row.get('OH7'),1,24)}

def build_label(baseline,followups):
    """Return y/reason/metadata. Never infer a NO from skip or unknown codes."""
    start=interview_date(baseline,'FINTDYR','FINTDMO','FINTDDY',1982,1984)
    if start is None:return None,'invalid_baseline_date',{}
    horizon=anniversary(start,HORIZON_YEARS)
    info={'baseline_date':start.isoformat(),'horizon_date':horizon.isoformat()}
    if integer(baseline.get('D2'))!=2:
        return None,'baseline_prior_diagnosis_or_unknown',info
    x=baseline_features(baseline)
    if any(not np.isfinite(x[k]) for k in FEATURES[:4]):
        return None,'baseline_age_sex_or_bp_missing',info
    # Cohort eligibility for this prototype, not a claim of clinical diagnosis.
    if x['baseline_sbp']>=130 or x['baseline_dbp']>=80:
        return None,'baseline_measured_bp_above_project_cutoff',info
    events=[]; no_dates=[]; death_years=[]; undated_yes=False; any_reported_yes=False
    for wave,row in followups.items():
        status=row.get('MAINRSLT','').strip() if wave=='92' else integer(row.get('BASERSLT'))
        usable=status in {'C1','C2','C3'} if wave=='92' else status in {1,2,3}
        if not usable:continue
        hyper=integer(row.get('HYPER'))
        if hyper not in {None,-1,-7,-8,-9,1,2,95,96}:
            return None,'unexpected_hypertension_code',info
        end=interview_date(row,'RSLTYR','RSLTMO','RSLTDY',1992,1993) if wave=='92' else (
            interview_date(row,'RSLTYR','RSLTMO','RSLTDAY',1987,1988) if wave=='87' else
            interview_date(row,'FINTYR','FINTMO','FINTDAY',1986,1987))
        if hyper==2 and end is not None:no_dates.append(end)
        death=integer(row.get('NEWDODYR' if wave=='92' else 'DEATHYR'))
        if death is not None:
            if 20<=death<=93:death+=1900
            if 1971<=death<=1993:death_years.append(death)
        if hyper in {95,96}:return None,'borderline_or_discrepant_report',info
        if hyper==1:
            any_reported_yes=True
            year=integer(row.get('HBPYEAR1'))
            limit=1993 if wave=='92' else 1988 if wave=='87' else 1987
            if year is None or not 20<=year<=limit-1900:
                undated_yes=True;continue
            year+=1900
            if year<=start.year:return None,'prior_or_same_year_diagnosis',info
            # A report cannot date first diagnosis beyond its interview year.
            if end is None or year>end.year:return None,'invalid_event_or_followup_date',info
            events.append(year)
    final=followups.get('92',{})
    final_status=final.get('MAINRSLT','').strip()
    final_hyper=integer(final.get('HYPER'))
    carried_yes=integer(final.get('LHIGHBP'))==1
    # In 1992 HYPER=-1 can mean an earlier positive was carried forward.
    # Require a dated positive in 1986/87 to resolve this; skip alone is not an event.
    if carried_yes and not any_reported_yes:undated_yes=True
    if final_status in {'C1','C2','C3'} and final_hyper==2 and any_reported_yes:
        return None,'contradictory_later_no_after_yes',info
    if final_status in {'C1','C2','C3'} and final_hyper==2 and carried_yes:
        return None,'contradictory_carried_history',info
    if len(set(events))>1:return None,'conflicting_first_diagnosis_years',info
    if undated_yes:return None,'positive_history_without_reliable_year',info
    if events:
        year=events[0]
        if any(year<=d.year for d in no_dates):
            return None,'diagnosis_year_conflicts_with_later_explicit_no',info
        if death_years and year>min(death_years):
            return None,'diagnosis_reported_after_death_year',info
        info['first_diagnosis_year']=year
        lower,upper=date(year,1,1),date(year,12,31)
        if upper<=horizon:return 1,'reported_diagnosis_within_8y',info
        if lower<=horizon:return None,'diagnosis_year_overlaps_horizon',info
        # Observed first diagnosis strictly after horizon establishes event-free
        # status at the horizon, conditional on the accuracy of the report.
        return 0,'first_reported_diagnosis_after_8y',info
    end=interview_date(final,'RSLTYR','RSLTMO','RSLTDY',1992,1993)
    if final_status not in {'C1','C2'}:
        return None,'no_completed_surviving_final_interview',info
    if end is None or end<horizon:
        return None,'followup_shorter_than_8y_or_invalid',info
    if final_hyper!=2 or carried_yes:
        return None,'final_target_unknown_or_skipped',info
    info['last_interview_date']=end.isoformat()
    return 0,'explicit_no_with_complete_8y_followup',info

def parse_layout(text):
    specs={n:(int(p)-1,int(p)-1+int(w)) for p,n,w in
           re.findall(r'@(\d+)\s+(\w+)\s+\$CHAR(\d+)\.',text,re.I)}
    if not specs:raise ValueError('Invalid SAS layout')
    return specs

def read_wave(path,layout,fields,idfield):
    missing=set(fields)-set(layout)
    if missing:raise ValueError(f'{path.name}: layout missing {missing}')
    rows={};width=max(layout[k][1] for k in fields)
    full_width=max(e for s,e in layout.values())
    with path.open(encoding='ascii') as f:
        for number,line in enumerate(f,1):
            line=line.rstrip('\r\n')
            if not line.strip():continue
            if len(line)<width or len(line)>full_width:
                raise ValueError(f'{path.name} record {number}: unexpected width {len(line)}')
            row={k:line[layout[k][0]:layout[k][1]].strip() for k in fields}
            ident=row[idfield]
            if not re.fullmatch(r'\d{5}',ident) or ident in rows:
                raise ValueError(f'{path.name}: invalid or duplicate ID on line {number}')
            rows[ident]=row
    if not rows:raise ValueError('No participant rows')
    return rows

def download(raw,name):
    path=raw/name
    if path.exists():return path
    print('Downloading',name,flush=True)
    temp=raw/(name+'.part')
    try:
        with urlopen(Request(BASE+name,headers={'User-Agent':'Mozilla/5.0'}),timeout=120) as r, temp.open('wb') as f:
            while chunk:=r.read(1024*1024):f.write(chunk)
        if temp.stat().st_size<100:raise ValueError('Empty source')
        with temp.open('rb') as f:head=f.read(100).lstrip().lower()
        if head.startswith((b'<!doctype',b'<html')):raise ValueError('HTML response instead of data')
        temp.replace(path)
    except Exception:
        if temp.exists():temp.unlink()
        raise
    return path

def write_csv(path,rows,columns):
    with path.open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=columns);w.writeheader();w.writerows(rows)

def prepare_cohort(root):
    raw=root/'raw';out=root/'outputs';raw.mkdir(parents=True,exist_ok=True);out.mkdir(exist_ok=True)
    waves={};manifest=[]
    for wave,config in WAVES.items():
        sources={n:download(raw,n) for n in [config['data'],config['layout']]}
        spec=parse_layout(sources[config['layout']].read_text(encoding='utf-8-sig'))
        anchors={'82':{'SEQNUM1':(0,5),'D2':(407,408),'PMR4cAVS':(4142,4145)},
                 '86':{'SEQNO':(0,5),'HYPER':(650,652)},
                 '87':{'SEQNO':(0,5),'HYPER':(756,758)},
                 '92':{'HANESEQ':(0,5),'HYPER':(750,752)}}
        for key,expected in anchors[wave].items():
            if spec.get(key)!=expected:raise ValueError(f'Official layout changed: {wave} {key}')
        waves[wave]=read_wave(sources[config['data']],spec,config['fields'],config['id'])
        for name,path in sources.items():
            manifest.append({'name':name,'url':BASE+name,'bytes':path.stat().st_size,
                             'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
    included=[];audit_rows=[];merged=[]
    for ident,b in sorted(waves['82'].items()):
        f={wave:rows[ident] for wave,rows in waves.items() if wave!='82' and ident in rows}
        y,reason,details=build_label(b,f)
        audit_rows.append({'participant_id':ident,'target':y,'reason':reason,**details})
        row={'participant_id':ident,**{'b82_'+k:v for k,v in b.items()}}
        for wave,follow in f.items():row.update({'f'+wave+'_'+k:v for k,v in follow.items()})
        merged.append(row)
        if y is not None:
            included.append({'participant_id':ident,**baseline_features(b),'target':y,**details})
    cols=sorted({k for row in merged for k in row})
    write_csv(out/'NHEFS_Merged_Four_Waves_Raw.csv',merged,cols)
    cols=sorted({k for row in audit_rows for k in row})
    write_csv(out/'NHEFS_Target_Audit.csv',audit_rows,cols)
    cols=['participant_id']+FEATURES+['target','baseline_date','horizon_date','first_diagnosis_year','last_interview_date']
    write_csv(out/'NHEFS_Future_8y_Cohort.csv',included,cols)
    audit={'wave_rows':{w:len(rows) for w,rows in waves.items()},'cohort_n':len(included),
      'target_counts':dict(Counter(str(r['target']) for r in included)),
      'all_reasons':dict(Counter(r['reason'] for r in audit_rows)),
      'sources':manifest,'data_origin':'official_CDC_raw','horizon_years':HORIZON_YEARS,'features':FEATURES,
      'target':'first reported physician diagnosis within 8 years',
      'limitations':['historical US cohort','self/proxy reported diagnosis; not measured future hypertension',
       'year-only event time; horizon-overlap cases excluded','unknown, lost and early deaths without known event excluded',
       'selection bias; no IPCW/competing-risk adjustment','not validated for modern Thai users',
       'sleep numeric physical-validity filtering; dietary raw codes retained but not modeled',
       'BMI and original 11-feature schema not available in this prototype']}
    (out/'NHEFS_Data_Audit.json').write_text(json.dumps(audit,indent=2,ensure_ascii=False),encoding='utf-8')
    if len(included)<500 or min(Counter(r['target'] for r in included).values(),default=0)<80 or len(set(r['target'] for r in included))!=2:
        raise ValueError('Insufficient reliable labels for this prototype. Inspect audit; do not relabel unknown as NO.')
    return included,audit

def sigmoid(z):
    z=np.asarray(z,dtype=float)
    return np.exp(-np.logaddexp(0.,-z))

def fit_logistic(x,y,ridge=1.0):
    x=np.asarray(x,float);y=np.asarray(y,float)
    a=np.column_stack([np.ones(len(x)),x]);coef=np.zeros(a.shape[1])
    penalty=np.eye(a.shape[1])*ridge;penalty[0,0]=0
    def objective(w):
        z=a@w
        return float(np.sum(np.logaddexp(0,z)-y*z)+.5*w@penalty@w)
    for iteration in range(100):
        p=sigmoid(a@coef)
        g=a.T@(p-y)+penalty@coef
        if np.max(np.abs(g))<1e-6:return coef.tolist()
        h=a.T@(a*(p*(1-p))[:,None])+penalty+np.eye(a.shape[1])*1e-9
        delta=np.linalg.solve(h,g);rate=1.;loss=objective(coef)
        while rate>1e-8 and objective(coef-rate*delta)>loss:rate*=.5
        if rate<=1e-8:raise ValueError('Logistic solver failed line search')
        coef-=rate*delta
    raise ValueError('Logistic solver failed to converge')

def logits(x,coef):
    return np.column_stack([np.ones(len(x)),np.asarray(x,float)])@np.asarray(coef,float)

def auc(y,p):
    y=np.asarray(y,int);p=np.asarray(p,float)
    npos=int(y.sum());nneg=len(y)-npos
    if not npos or not nneg:return None
    order=np.argsort(p);ranks=np.empty(len(y),float);i=0
    while i<len(y):
        j=i+1
        while j<len(y) and p[order[j]]==p[order[i]]:j+=1
        ranks[order[i:j]]=(i+1+j)/2;i=j
    return float((ranks[y==1].sum()-npos*(npos+1)/2)/(npos*nneg))

def metrics(y,p):
    y=np.asarray(y,int);p=np.clip(np.asarray(p,float),1e-12,1-1e-12);pred=p>=.5
    tp=int(np.sum(pred&(y==1)));fp=int(np.sum(pred&(y==0)))
    tn=int(np.sum(~pred&(y==0)));fn=int(np.sum(~pred&(y==1)))
    precision=tp/(tp+fp) if tp+fp else 0.
    recall=tp/(tp+fn) if tp+fn else 0.
    specificity=tn/(tn+fp) if tn+fp else 0.
    return {'n':len(y),'events':int(y.sum()),'roc_auc':auc(y,p),'brier_score':float(np.mean((p-y)**2)),
      'log_loss':float(-np.mean(y*np.log(p)+(1-y)*np.log1p(-p))),
      'accuracy':float(np.mean(pred==y)),'precision':precision,'recall':recall,
      'balanced_accuracy':(recall+specificity)/2,'f1':2*precision*recall/(precision+recall) if precision+recall else 0.,
      'confusion_matrix':{'tn':tn,'fp':fp,'fn':fn,'tp':tp},'threshold':.5,
      'observed_rate':float(np.mean(y)),'mean_predicted':float(np.mean(p))}

def split_ids(y):
    rng=np.random.default_rng(SEED);splits={k:[] for k in ['train','tune','calibration','test']}
    for label in [0,1]:
        indices=np.where(np.asarray(y)==label)[0];rng.shuffle(indices)
        n=len(indices);ends=[int(n*.55),int(n*.70),int(n*.85)]
        for name,part in zip(splits,np.split(indices,ends)):splits[name].extend(part.tolist())
    splits={k:np.array(sorted(v),int) for k,v in splits.items()}
    flat=[i for v in splits.values() for i in v]
    assert len(flat)==len(set(flat))==len(y)
    return splits

def fit_preprocessor(x):
    x=np.asarray(x,float)
    if np.any(np.all(np.isnan(x),axis=0)):raise ValueError('A feature is entirely missing in Train')
    med=np.nanmedian(x,axis=0);filled=np.where(np.isnan(x),med,x)
    mean=filled.mean(axis=0);scale=filled.std(axis=0);scale[scale<1e-12]=1
    return {'median':med.tolist(),'mean':mean.tolist(),'scale':scale.tolist()}

def transform(x,prep):
    x=np.asarray(x,float)
    return (np.where(np.isnan(x),prep['median'],x)-prep['mean'])/prep['scale']

def train_model(cohort,root,audit):
    out=root/'outputs'
    x=np.array([[row[f] for f in FEATURES] for row in cohort],float)
    y=np.array([row['target'] for row in cohort],int);splits=split_ids(y)
    prep=fit_preprocessor(x[splits['train']]);z=transform(x,prep)
    assert np.isfinite(z).all()
    candidates=[]
    for ridge in [.01,.1,1.,10.,100.]:
        coef=fit_logistic(z[splits['train']],y[splits['train']],ridge)
        p=sigmoid(logits(z[splits['tune']],coef))
        candidates.append({'ridge':ridge,'coef':coef,'tune_brier':metrics(y[splits['tune']],p)['brier_score']})
    winner=min(candidates,key=lambda c:c['tune_brier'])
    cids=splits['calibration'];score=logits(z[cids],winner['coef'])
    cal=fit_logistic(score[:,None],y[cids],ridge=.1)
    if cal[1]<=0:raise ValueError('Calibration slope nonpositive; review data and model before using demo')
    bundle={'format_version':1,'model':'ridge_logistic_with_platt_calibration',
      'horizon_years':HORIZON_YEARS,'features':FEATURES,'preprocessor':prep,
      'coefficients':winner['coef'],'calibration_coefficients':cal,'ridge':winner['ridge'],
      'training_range':{f:[float(np.nanmin(x[splits['train'],j])),float(np.nanmax(x[splits['train'],j]))] for j,f in enumerate(FEATURES)},
      'target':audit['target'],'limitations':audit['limitations'],
      'trained_on_real_nhefs':audit.get('data_origin')=='official_CDC_raw'}
    t=splits['test'];score_test=logits(z[t],winner['coef'])
    probability=sigmoid(cal[0]+cal[1]*score_test)
    constant=np.full(len(t),y[splits['train']].mean())
    results={'selected_ridge':winner['ridge'],'tuning':[{'ridge':c['ridge'],'brier':c['tune_brier']} for c in candidates],
      'splits':{k:{'n':len(v),'events':int(y[v].sum())} for k,v in splits.items()},
      'test_model':metrics(y[t],probability),'test_constant_baseline':metrics(y[t],constant),
      'test_uncalibrated':metrics(y[t],sigmoid(score_test))}
    # Independent held-out test; uncertainty resampling does not retune the model.
    rng=np.random.default_rng(SEED);samples=[]
    for _ in range(200):
        ids=rng.integers(0,len(t),len(t));a=auc(y[t][ids],probability[ids])
        if a is not None:samples.append(a)
    results['test_auc_bootstrap_95ci']=[float(v) for v in np.quantile(samples,[.025,.975])]
    calibration=[]
    for bin_id in range(10):
        lo=bin_id/10
        mask=np.minimum((probability*10).astype(int),9)==bin_id
        if mask.any():calibration.append({'bin_lower':round(float(lo),1),'n':int(mask.sum()),
         'mean_predicted':float(probability[mask].mean()),'observed_rate':float(y[t][mask].mean())})
    write_csv(out/'NHEFS_Test_Calibration.csv',calibration,['bin_lower','n','mean_predicted','observed_rate'])
    write_csv(out/'NHEFS_Test_Predictions.csv',[{'participant_id':cohort[i]['participant_id'],'target':int(y[i]),
      'probability_8y':float(p)} for i,p in zip(t,probability)],['participant_id','target','probability_8y'])
    split_by={i:name for name,v in splits.items() for i in v}
    write_csv(out/'NHEFS_Model_Data.csv',[{'participant_id':row['participant_id'],**{f:row[f] for f in FEATURES},
      'target':row['target'],'split':split_by[i]} for i,row in enumerate(cohort)],['participant_id']+FEATURES+['target','split'])
    (out/'NHEFS_Future_Model.json').write_text(json.dumps(bundle,indent=2,allow_nan=False),encoding='utf-8')
    (out/'NHEFS_Model_Metrics.json').write_text(json.dumps(results,indent=2,allow_nan=False),encoding='utf-8')
    feature_info=[{'feature':f,'source':'1982-1984 '+s,'unit':u,'role':'baseline_predictor'} for f,s,u in
       zip(FEATURES,['SUBAGE','SUBSEX','PMR4cAVS or mean 2/3','PMR4cAVD or mean 2/3','OH7'],
           ['years','0=male 1=female','mmHg','mmHg','hours/night'])]
    write_csv(out/'NHEFS_Feature_Dictionary.csv',feature_info,['feature','source','unit','role'])
    report=f'''# LEPTOLIC — NHEFS future prediction prototype

Target: first **reported physician diagnosis** of hypertension within 8 years after
the 1982–1984 interview. This is a future outcome, not current hypertension.
The diagnosis is self/proxy reported, not verified clinical onset.
Baseline: explicit D2=2 and measured SBP<130 / DBP<80 under this project's rule.
Sex, age, baseline pressure and optional sleep are the only model inputs.
Dietary raw columns are retained for inspection; BMI, sodium mg and the old 11-feature
schema have NOT been recreated or claimed equivalent.

Reliable-labeled cohort: {len(cohort):,}. Labels: {audit['target_counts']}.
Source rows: {audit['wave_rows']}.
Train/tune/calibration/test: {results['splits']}.
Held-out test ROC-AUC: {results['test_model']['roc_auc']:.3f}
Test Brier: {results['test_model']['brier_score']:.4f}
Constant-baseline Brier: {results['test_constant_baseline']['brier_score']:.4f}
These are internal historical results, not modern-population validation.

Year-only event dates define an interval, not a fabricated day: if a diagnosis year
overlaps the 8-year horizon, its label is excluded. Inapplicable, don't know, refused,
borderline, discrepant, undated positives, inconsistent years and insufficient follow-up
are excluded with reasons in Target_Audit.csv. A skipped HYPER is never a negative.
A known earlier positive with reliable year can establish an event even if the person
later died or was lost; an early death/loss without known event is never relabeled NO.
The labeled cohort can still suffer informative-censoring and selection bias; no
population-risk claim, IPCW or competing-risk correction is made.

Preprocessing fitted only on Train. Ridge selected only on Tune. Platt calibration
fitted only on Calibration. Test used only for evaluation, never selection or refitting.
Dummy baseline uses Train event prevalence. Seed={SEED}. Threshold=0.5 is a demo
threshold, not a clinical threshold. No survey-weighted national estimates are reported.

Historical US cohort, not validated for modern Thailand. Model probabilities are
experimental scores conditional on the selected labeled sample. Not medical advice
or a clinically validated personal risk estimate. Out-of-range ages/pressures are rejected.

CDC documentation:
https://wwwn.cdc.gov/nchs/nhanes/nhefs/default.aspx
https://wwwn.cdc.gov/nchs/data/nhefs/1982intv.pdf
https://wwwn.cdc.gov/nchs/data/nhefs/1986intv.pdf
https://wwwn.cdc.gov/nchs/data/nhefs/1987intv.pdf
https://wwwn.cdc.gov/nchs/data/nhefs/1992intv.pdf
Source: National Center for Health Statistics (NHEFS).
Analyses and interpretations are those of the author, not NCHS.
'''
    (out/'NHEFS_Future_Report.md').write_text(report,encoding='utf-8')
    archive=root/'LEPTOLIC_NHEFS_Future_Outputs.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as f:
        for p in sorted(out.iterdir()):f.write(p,p.name)
    print(json.dumps(results,indent=2))
    return bundle,results,archive

def predict_future(bundle,values,previously_diagnosed=False):
    if previously_diagnosed:raise ValueError('Model only applies to people without a prior reported diagnosis at baseline')
    if bundle.get('trained_on_real_nhefs') is not True:raise ValueError('No real NHEFS trained model')
    row=[];warnings=[]
    for f in bundle['features']:
        value=values.get(f)
        if value is None or (isinstance(value,float) and math.isnan(value)):
            if f!='sleep_hours':raise ValueError(f'Missing required input: {f}')
            row.append(np.nan);warnings.append('Missing sleep imputed from Train');continue
        value=float(value)
        if not math.isfinite(value):raise ValueError('Input must be finite')
        lo,hi=bundle['training_range'][f]
        if value<lo or value>hi:raise ValueError(f'{f} outside training range [{lo:g}, {hi:g}]')
        if f=='sex_female' and value not in {0,1}:raise ValueError('sex_female must be 0 or 1')
        row.append(value)
    if values['baseline_sbp']>=130 or values['baseline_dbp']>=80:
        raise ValueError('Baseline pressure outside this prototype cohort eligibility')
    z=transform([row],bundle['preprocessor']);s=float(logits(z,bundle['coefficients'])[0])
    intercept,slope=bundle['calibration_coefficients']
    return {'experimental_probability_8y':float(sigmoid(intercept+slope*s)),
      'endpoint':bundle['target'],'horizon_years':8,'warnings':warnings,
      'use':'Historical research/demo only; not validated modern Thai risk'}

def run_pipeline(root=None):
    root=Path(root or ('/content/leptolic_nhefs_future' if Path('/content').exists() else 'nhefs_future_work'))
    cohort,audit=prepare_cohort(root)
    bundle,results,archive=train_model(cohort,root,audit)
    return root,cohort,bundle,results,archive


## 1. ดาวน์โหลด รวมข้อมูล และสร้าง Target

In [ ]:
# ดาวน์โหลดและเตรียมข้อมูลจริง หากล้มเหลวจะหยุด ไม่แอบใช้ข้อมูลจำลอง
ROOT = Path('/content/leptolic_nhefs_future')
cohort, data_audit = prepare_cohort(ROOT)
print('Reliable-labeled cohort:', len(cohort))
print('Targets:', data_audit['target_counts'])
print('Excluded/retained reasons:')
print(json.dumps(data_audit['all_reasons'], ensure_ascii=False, indent=2))


Reliable-labeled cohort: 2720
Targets: {'0': 2554, '1': 166}
Excluded/retained reasons:
{
  "baseline_measured_bp_above_project_cutoff": 2930,
  "baseline_prior_diagnosis_or_unknown": 4748,
  "explicit_no_with_complete_8y_followup": 2506,
  "baseline_age_sex_or_bp_missing": 1168,
  "prior_or_same_year_diagnosis": 69,
  "diagnosis_year_conflicts_with_later_explicit_no": 20,
  "no_completed_surviving_final_interview": 462,
  "borderline_or_discrepant_report": 37,
  "reported_diagnosis_within_8y": 166,
  "first_reported_diagnosis_after_8y": 48,
  "positive_history_without_reliable_year": 32,
  "diagnosis_year_overlaps_horizon": 26,
  "final_target_unknown_or_skipped": 6,
  "conflicting_first_diagnosis_years": 1,
  "contradictory_later_no_after_yes": 1
}


## 2. ฝึก ปรับ probability และประเมิน

ใช้ Logistic Regression พร้อม L2 regularization และ Platt calibration
เปรียบเทียบกับ constant baseline ที่ใช้สัดส่วนเหตุการณ์จาก Train
แสดง ROC-AUC, Brier score, Precision, Recall, F1, Accuracy และ Calibration
รายงานทุกคะแนนตามจริง แม้โมเดลจะไม่ชนะ baseline


In [ ]:
# Train 55% / Tune 15% / Calibration 15% / Test 15% แบบ stratified
# ค่า preprocessing เรียนจาก Train เท่านั้น และไม่ใช้ Test เลือก/ปรับโมเดล
model_bundle, model_metrics, output_zip = train_model(cohort, ROOT, data_audit)
from IPython.display import display, Markdown
display(Markdown((ROOT/'outputs'/'NHEFS_Future_Report.md').read_text(encoding='utf-8')))


{
  "selected_ridge": 10.0,
  "tuning": [
    {
      "ridge": 0.01,
      "brier": 0.056125802007497484
    },
    {
      "ridge": 0.1,
      "brier": 0.0561244407983877
    },
    {
      "ridge": 1.0,
      "brier": 0.0561119289225206
    },
    {
      "ridge": 10.0,
      "brier": 0.056060791179222275
    },
    {
      "ridge": 100.0,
      "brier": 0.05648746498991715
    }
  ],
  "splits": {
    "train": {
      "n": 1495,
      "events": 91
    },
    "tune": {
      "n": 408,
      "events": 25
    },
    "calibration": {
      "n": 408,
      "events": 25
    },
    "test": {
      "n": 409,
      "events": 25
    }
  },
  "test_model": {
    "n": 409,
    "events": 25,
    "roc_auc": 0.6761458333333333,
    "brier_score": 0.056529185516960026,
    "log_loss": 0.2227443553335793,
    "accuracy": 0.9388753056234719,
    "precision": 0.0,
    "recall": 0.0,
    "balanced_accuracy": 0.5,
    "f1": 0.0,
    "confusion_matrix": {
      "tn": 384,
      "fp": 0,
      "fn": 25,
 

# LEPTOLIC — NHEFS future prediction prototype

Target: first **reported physician diagnosis** of hypertension within 8 years after
the 1982–1984 interview. This is a future outcome, not current hypertension.
The diagnosis is self/proxy reported, not verified clinical onset.
Baseline: explicit D2=2 and measured SBP<130 / DBP<80 under this project's rule.
Sex, age, baseline pressure and optional sleep are the only model inputs.
Dietary raw columns are retained for inspection; BMI, sodium mg and the old 11-feature
schema have NOT been recreated or claimed equivalent.

Reliable-labeled cohort: 2,720. Labels: {'0': 2554, '1': 166}.
Source rows: {'82': 12220, '86': 3608, '87': 9998, '92': 9281}.
Train/tune/calibration/test: {'train': {'n': 1495, 'events': 91}, 'tune': {'n': 408, 'events': 25}, 'calibration': {'n': 408, 'events': 25}, 'test': {'n': 409, 'events': 25}}.
Held-out test ROC-AUC: 0.676
Test Brier: 0.0565
Constant-baseline Brier: 0.0574
These are internal historical results, not modern-population validation.

Year-only event dates define an interval, not a fabricated day: if a diagnosis year
overlaps the 8-year horizon, its label is excluded. Inapplicable, don't know, refused,
borderline, discrepant, undated positives, inconsistent years and insufficient follow-up
are excluded with reasons in Target_Audit.csv. A skipped HYPER is never a negative.
A known earlier positive with reliable year can establish an event even if the person
later died or was lost; an early death/loss without known event is never relabeled NO.
The labeled cohort can still suffer informative-censoring and selection bias; no
population-risk claim, IPCW or competing-risk correction is made.

Preprocessing fitted only on Train. Ridge selected only on Tune. Platt calibration
fitted only on Calibration. Test used only for evaluation, never selection or refitting.
Dummy baseline uses Train event prevalence. Seed=42. Threshold=0.5 is a demo
threshold, not a clinical threshold. No survey-weighted national estimates are reported.

Historical US cohort, not validated for modern Thailand. Model probabilities are
experimental scores conditional on the selected labeled sample. Not medical advice
or a clinically validated personal risk estimate. Out-of-range ages/pressures are rejected.

CDC documentation:
https://wwwn.cdc.gov/nchs/nhanes/nhefs/default.aspx
https://wwwn.cdc.gov/nchs/data/nhefs/1982intv.pdf
https://wwwn.cdc.gov/nchs/data/nhefs/1986intv.pdf
https://wwwn.cdc.gov/nchs/data/nhefs/1987intv.pdf
https://wwwn.cdc.gov/nchs/data/nhefs/1992intv.pdf
Source: National Center for Health Statistics (NHEFS).
Analyses and interpretations are those of the author, not NCHS.


## 3. ทดลองทำนายจากข้อมูลตั้งต้น

In [ ]:
# ตัวอย่างจากชุดข้อมูลจริงที่อยู่ในขอบเขตการฝึก ปรับค่าแล้วรันเซลล์นี้ใหม่ได้
demo_row = next(row for row in cohort if all(
    (not np.isfinite(row[f]) and f == 'sleep_hours') or
    (model_bundle['training_range'][f][0] <= row[f] <= model_bundle['training_range'][f][1])
    for f in FEATURES))
INPUT = {f: float(demo_row[f]) for f in FEATURES}
# ตัวอย่างการแก้ INPUT:
# INPUT['age_years'] = 50
# INPUT['sex_female'] = 0  # 0=ชาย 1=หญิง ตามตัวแปรเพศในแบบสำรวจ
# INPUT['baseline_sbp'] = 120
# INPUT['baseline_dbp'] = 75
# INPUT['sleep_hours'] = 7
PREVIOUSLY_DIAGNOSED = False
answer = predict_future(model_bundle, INPUT, previously_diagnosed=PREVIOUSLY_DIAGNOSED)
print('Inputs:', INPUT)
print(f"คะแนนทดลองสำหรับผลรายงานวินิจฉัยใน 8 ปี: {answer['experimental_probability_8y']:.1%}")
print('ผลนี้ใช้สาธิตจากข้อมูลอดีต ยังไม่ใช่ความเสี่ยงที่ยืนยันสำหรับคนไทยยุคนี้')
print('Warnings:', answer['warnings'])


Inputs: {'age_years': 47.0, 'sex_female': 1.0, 'baseline_sbp': 100.0, 'baseline_dbp': 70.0, 'sleep_hours': nan}
คะแนนทดลองสำหรับผลรายงานวินิจฉัยใน 8 ปี: 4.9%
ผลนี้ใช้สาธิตจากข้อมูลอดีต ยังไม่ใช่ความเสี่ยงที่ยืนยันสำหรับคนไทยยุคนี้
Warnings: ['Missing sleep imputed from Train']


In [ ]:
# แบบฟอร์มสาธิตใน Colab (หาก widgets ไม่พร้อม ใช้ INPUT เซลล์ด้านบนแทนได้)
try:
    import ipywidgets as widgets
    controls = {}
    titles = {'age_years':'อายุ (ปี)','baseline_sbp':'ค่าบน (mmHg)',
              'baseline_dbp':'ค่าล่าง (mmHg)','sleep_hours':'นอน (ชั่วโมง)'}
    for f in FEATURES:
        if f == 'sex_female':
            controls[f]=widgets.Dropdown(options=[('ชาย',0),('หญิง',1)],value=int(INPUT[f]),description='เพศ:')
        else:
            default=INPUT[f] if np.isfinite(INPUT[f]) else model_bundle['preprocessor']['median'][FEATURES.index(f)]
            controls[f]=widgets.FloatText(value=default,description=titles[f]+':',style={'description_width':'initial'})
    prior=widgets.Checkbox(value=False,description='เคยได้รับการวินิจฉัยความดันสูงแล้ว')
    button=widgets.Button(description='คำนวณคะแนนทดลอง',button_style='info')
    output=widgets.Output()
    def calculate(_):
        with output:
            output.clear_output()
            try:
                result=predict_future(model_bundle,{k:v.value for k,v in controls.items()},prior.value)
                print(f"คะแนนทดลอง 8 ปี: {result['experimental_probability_8y']:.1%}")
                print('ข้อมูลอดีตสหรัฐฯ — ยังไม่ใช่ผลความเสี่ยงที่ยืนยันสำหรับคนไทยยุคนี้')
            except ValueError as exc:
                print('ยังคำนวณไม่ได้:',exc)
    button.on_click(calculate)
    display(widgets.VBox(list(controls.values())+[prior,button,output]))
except ImportError:
    print('ใช้ INPUT เซลล์ด้านบนเพื่อทำนายแทนแบบฟอร์มได้')


## 4. ดาวน์โหลดผล

ZIP มีข้อมูลที่รวมแล้ว, audit การคัด label, ชุดข้อมูลพร้อม split, โมเดล JSON,
ผล Test, ตาราง Calibration, dictionary และรายงานข้อจำกัด
ไม่มีข้อมูล NHANES ใหม่หรือฟีเจอร์อาหารที่แต่งขึ้นมาแทนค่าที่ไม่มี
